# 01 – SFT: Denkmuster lernen
Kaggle: **GPU T4 x2**, **Internet an**. Dauer bei ~3k Beispielen: grob 1–3 h.

In [ ]:
# === Setup: Repo klonen + Abhängigkeiten ===
REPO_URL = "https://github.com/Lommel123/creativ-model.git"   # <- anpassen

import os, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass  # öffentliches Repo oder kein Secret gesetzt

if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
%pip install -q unsloth pyyaml
import yaml
CFG = yaml.safe_load(open("repo/config.yaml"))
CFG

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

def chat(tokenizer, messages, add_generation_prompt=False):
    kw = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kw)  # Qwen3: Thinking-Modus aus
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kw)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG["base_model"], max_seq_length=CFG["max_seq_length"], load_in_4bit=True)
L = CFG["lora"]
model = FastLanguageModel.get_peft_model(
    model, r=L["r"], lora_alpha=L["alpha"], lora_dropout=L["dropout"],
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42)

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files="repo/data/sft.jsonl", split="train")
ds = ds.map(lambda r: {"text": chat(tokenizer, r["messages"])})
print(len(ds)); print(ds[0]["text"][:600])

In [ ]:
from trl import SFTTrainer, SFTConfig
S = CFG["sft"]
trainer = SFTTrainer(
    model=model, processing_class=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=CFG["max_seq_length"],
        per_device_train_batch_size=S["batch_size"], gradient_accumulation_steps=S["grad_accum"],
        num_train_epochs=S["epochs"], learning_rate=S["lr"], warmup_steps=max(1, int(0.05 * len(ds) * S["epochs"] / (S["batch_size"] * S["grad_accum"]))),
        lr_scheduler_type="cosine", optim="adamw_8bit", weight_decay=0.01,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=1, save_steps=200, save_total_limit=2,
        output_dir="checkpoints_sft", report_to="none", seed=42))
# Loss nur auf die Antworten, nicht auf die Fragen
from unsloth.chat_templates import train_on_responses_only
if "qwen" in CFG["base_model"].lower():
    parts = ("<|im_start|>user\n", "<|im_start|>assistant\n")
else:  # Llama 3.x
    parts = ("<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n")
trainer = train_on_responses_only(trainer, instruction_part=parts[0], response_part=parts[1])
trainer.train()

In [ ]:
# Schnelltest
FastLanguageModel.for_inference(model)
p = chat(tokenizer, [{"role":"user","content":"Wofür kann man einen Ziegelstein noch nutzen?"}], add_generation_prompt=True)
ids = tokenizer(p, return_tensors="pt").to("cuda")
print(tokenizer.decode(model.generate(**ids, max_new_tokens=300, temperature=0.8, do_sample=True)[0][ids.input_ids.shape[1]:], skip_special_tokens=True))

In [ ]:
model.save_pretrained("adapter_sft"); tokenizer.save_pretrained("adapter_sft")
print("Gespeichert in /kaggle/working/adapter_sft → jetzt 'Save Version' klicken")